In [1]:
print(123)

123


In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from openai import OpenAI
openai_client = OpenAI()

In [6]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [8]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes — you may be able to join, depending on the course’s enrollment policy and whether it’s still open.

If you want, I can help you figure it out. Please send:
- the course name
- the school/platform
- the current date or deadline, if you know it
- whether you mean “join” as in enroll, attend live, or catch up on missed work

If you’re asking generally, the best next step is to check:
1. whether registration is still open,
2. if there’s a waitlist or late-enrollment option,
3. whether the instructor can approve a late join.

If you want, I can also draft a short message asking the instructor if you can still join.


In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [23]:
question = "I just discovered the course. Can I join now?"

In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

NameError: name 'question' is not defined

In [ ]:
answer = llm(prompt)
print(answer)

In [9]:
import requests

In [10]:
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [20]:
documents[500]

{'id': 'b5d37f88b6',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How are homework assignments scored?',
 'answer': "Homework grade = points for the questions + 1 point for the FAQ + up to 7 points for learning in public.\n\n- Learning in public: up to 7 points, depending on how many platforms you shared your post on (e.g. LinkedIn, X, blog) and the quality of the post.\n- FAQ: 1 point. To get it, contribute to the FAQ repo and add the link to your PR in your homework submission.\n\nOptional questions are scored too - read the submission form carefully. The homework is for practice and does not affect your certificate. If you think you weren't graded, log in to check your results or search for your name on the leaderboard."}

In [ ]:
# Index data to get most relevant (lucene, elasticsearch, solr, minsearch)
# Minsearch is a lightweight similar to elasticsearch
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'] # keyword_fields must be an exact match
)

index.fit(documents)

In [30]:
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section':0.5},
    filter_dict={'course': 'llm-zoomcamp'}, 
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [33]:
def search(question, course='llm-zoomcamp'):
    boost_dict={'question': 2.0, 'section':0.5}
    filter_dict={'course': course}
    
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict, 
        num_results=5
    )

In [3]:
search_results = search(question)
search_results

NameError: name 'search' is not defined

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [ ]:
USER_PROMPT_TEMPLATE = f"""
Question:
{question}

Context:
{context}
"""

In [1]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

NameError: name 'search_results' is not defined

In [ ]:
prompt = build_prompt(question, search_results)
print(prompt)

NameError: name 'USER_PROMPT_TEMPLATE' is not defined